In [6]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:95%;}

div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li{font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:40px;}
</style>
"""))

# 5. 생성형 AI 평가 : 
- 첫번째 체인 :  나라이름 -> 그 나라에서 가장 유명한 음식 
- 두번째 체인 :  음식 -> 음식의 레시피
- 최종 체인 : 나라이름 -> 그 나라에 가장 유명한 음식의 레시피

In [15]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser  #aimessage만 출력하는 parse
from langchain_core.output_parsers import JsonOutputParser
llm= ChatOllama(model="exaone3.5:2.4b")

#첫번째 체인 :  나라이름 -> 그 나라에서 가장 유명한 음식 
food_prompt_template = PromptTemplate(
    template="{국가}에서 가장 유명한 음식이 무엇입니까? 출력은 음식이름만! 출력해줘.",
    input_variables =['국가'])
outputParser=StrOutputParser()

In [16]:
outputParser.invoke(llm.invoke(food_prompt_template.invoke({'한국'})))

'비빔밥'

In [17]:
famous_food_chain= food_prompt_template | llm | outputParser
famous_food_chain.invoke({'국가':'한국'})

'김치'

In [18]:
#두번째 체인 :  음식 -> 음식의 레시피
recipe_prompt_template = PromptTemplate(
                    template="""다음 음식({음식이름})에 대한 정보를 제공해줘,
                            1. 음식 이름
                            2. 재료 준비 (Prep)
                            3. 조리 (Cook)
                            4. 마무리 (Finish) 
                추가 텍스트 없이 오직 유효한 JSON 객체만 반환해. 
                예시 형식:
                {{
                      "음식이름": "{음식이름}",
                      "1.재료 준비 (Prep)": "채소를 얇게 채 썬다.",
                      "2.조리 (Cook)": "채소와 소고기를 각각 볶은 뒤 계란 프라이를 만든다.",
                      "3.마무리 (Finish) ": "따뜻한 밥 위에 재료들과 계란 프라이, 고추장, 참기름을 올린다."
                }}""",
                    input_variables =['음식이름']
                    )

In [19]:
recipe_output_parser=JsonOutputParser()
recipe_output_parser.invoke(llm.invoke(recipe_prompt_template.invoke({'음식이름':'김치'})))

{'음식이름': '김치',
 '1.재료 준비 (Prep)': '배추를 소금물에 절이고, 고춧가루, 마늘, 생강, 젓갈 등을 섞어 양념을 만든다.',
 '2.조리 (Cook)': '절인 배추를 물기를 제거하고, 양념을 골고루 발라 눌러준다. 발효 시간을 거쳐 숙성시킨다.',
 '3.마무리 (Finish)': '익힌 고기나 해산물과 함께 구워 넣거나, 밥 위에 올려 먹는다.'}

In [20]:
recipe_chain= recipe_prompt_template | llm | recipe_output_parser
recipe_chain.invoke({'음식이름':'김치'})

{'음식이름': '김치',
 '1.재료 준비 (Prep)': '배추를 소금물에 절여 씨를 제거한 후, 고춧가루, 마늘, 생강 등 양념을 넣고 섞어 발효시킵니다.',
 '2.조리 (Cook)': '준비된 배추와 양념을 통기에서 잘 섞어 발효시킨 후, 최종적으로 소금을 넣어 간을 맞춥니다.',
 '3.마무리 (Finish)': '신선한 채소와 함께 구워낸 고기나 해산물과 함께 제공하거나, 밥 위에 올려 고추장과 참기름을 곁들여 먹습니다.'}

In [22]:
#최종 체인 : 나라이름 -> 그 나라에 가장 유명한 음식의 레시피
final_chain = famous_food_chain | recipe_chain
final_chain.invoke({'국가':'베트남'})

{'음식이름': '베트남 요리 중 가장 유명한 음식: 퍼 홍 (Pho)',
 '1.재료 준비 (Prep)': '소고기 broth (Phin Pho): 육수, 채소 (예: 양파, 마늘, 허브), 쌀 (주로 퍼 Pho rice)',
 '2.조리 (Cook)': '육수를 끓이고, 얇게 썬 소고기와 채소를 넣어 익히며, rice를 별도로 준비한다. 계란 프라이도 함께 준비한다',
 '3.마무리 (Finish)': '뜨거운 쌀 위에 소고기 broth를 부어 따뜻하게 즐기며, 얇게 썬 채소와 계란 프라이를 뿌리고, 고추장 (Nho)과 참기름 (Thaipho)을 추가해 맛을 더한다.'}